# MIS 433 Final Project: AI Investment Signals

This notebook explores whether historical stock trends, volatility, trading volume, and recent news sentiment can help forecast short-term investment signals among leading AI infrastructure companies.

The companies analyzed are NVDA, MSFT, GOOGL, AMZN, AMD, and AVGO. Instead of trying to predict exact future stock prices, this project builds toward a recommendation-support model that predicts whether a stock is likely to move up or down over the next 7 trading days.


## 1. Environment Setup

This section connects Google Drive, sets the working folder, installs the required Python packages, and imports the libraries used throughout the project.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content/drive/MyDrive/MIS_433_Project


In [ ]:
!mkdir -p data output colab_notebooks


In [ ]:
!pip install yfinance pandas numpy matplotlib seaborn scikit-learn requests google-generativeai


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_theme(style="whitegrid")


## 2. Stock Data Collection

We use `yfinance` to pull historical stock data from Yahoo Finance. The raw data includes open, high, low, close, adjusted close, and volume for each selected company.


In [ ]:
tickers = ["NVDA", "MSFT", "GOOGL", "AMZN", "AMD", "AVGO"]
start_date = "2020-01-01"
end_date = datetime.today().strftime("%Y-%m-%d")


In [ ]:
stock_data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    group_by="ticker"
)

stock_data.head()


In [ ]:
stock_data.to_csv("data/stock_prices_raw.csv")


## 3. Data Cleaning

The raw `yfinance` output is converted from a wide format into a cleaner long format where each row represents one company on one trading day.


In [ ]:
clean_data = []

for ticker in tickers:
    temp = stock_data[ticker].copy()
    temp["ticker"] = ticker
    temp = temp.reset_index()
    clean_data.append(temp)

stock_df = pd.concat(clean_data, ignore_index=True)
stock_df.columns = [col.lower().replace(" ", "_") for col in stock_df.columns]
stock_df["date"] = pd.to_datetime(stock_df["date"])

stock_df.head()


In [ ]:
stock_df.to_csv("data/stock_prices_clean.csv", index=False)


## 4. Exploratory Data Analysis

These charts and summary statistics help compare each company's stock price behavior over time.


In [ ]:
plt.figure(figsize=(12, 6))

for ticker in tickers:
    temp = stock_df[stock_df["ticker"] == ticker]
    plt.plot(temp["date"], temp["close"], label=ticker)

plt.title("Historical Closing Prices for AI Infrastructure Stocks")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
stock_df.info()
stock_df.describe()


## 5. Feature Engineering

The model uses calculated features such as daily returns, 7-day and 30-day returns, moving averages, 30-day volatility, and volume change.


In [ ]:
stock_df = stock_df.sort_values(["ticker", "date"])

stock_df["daily_return"] = stock_df.groupby("ticker")["close"].pct_change()
stock_df["return_7d"] = stock_df.groupby("ticker")["close"].pct_change(7)
stock_df["return_30d"] = stock_df.groupby("ticker")["close"].pct_change(30)

stock_df["ma_7d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=7).mean())
stock_df["ma_30d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=30).mean())
stock_df["ma_90d"] = stock_df.groupby("ticker")["close"].transform(lambda x: x.rolling(window=90).mean())

stock_df["volatility_30d"] = stock_df.groupby("ticker")["daily_return"].transform(lambda x: x.rolling(window=30).std())
stock_df["volume_change"] = stock_df.groupby("ticker")["volume"].pct_change()

stock_df.head()


In [ ]:
stock_df.to_csv("data/stock_features.csv", index=False)


## 6. Stock Performance Comparisons

A normalized chart makes it easier to compare companies with very different stock prices. Each company starts at an indexed value of 100.


In [ ]:
normalized_df = stock_df.copy()
normalized_df["normalized_close"] = normalized_df.groupby("ticker")["close"].transform(lambda x: x / x.iloc[0] * 100)

plt.figure(figsize=(12, 6))

for ticker in tickers:
    temp = normalized_df[normalized_df["ticker"] == ticker]
    plt.plot(temp["date"], temp["normalized_close"], label=ticker)

plt.title("Normalized Stock Performance Since 2020")
plt.xlabel("Date")
plt.ylabel("Indexed Price: 2020 = 100")
plt.legend()
plt.tight_layout()
plt.savefig("output/normalized_stock_performance.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
volatility_summary = stock_df.groupby("ticker")["daily_return"].std().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
volatility_summary.plot(kind="bar")
plt.title("Volatility Comparison by Stock")
plt.xlabel("Ticker")
plt.ylabel("Standard Deviation of Daily Returns")
plt.tight_layout()
plt.savefig("output/volatility_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
summary = stock_df.groupby("ticker").agg(
    start_price=("close", "first"),
    end_price=("close", "last"),
    avg_daily_return=("daily_return", "mean"),
    volatility=("daily_return", "std"),
    avg_volume=("volume", "mean")
).reset_index()

summary["total_return"] = (summary["end_price"] / summary["start_price"]) - 1
summary = summary.sort_values("total_return", ascending=False)

summary


## 7. News Sentiment Data from Alpha Vantage

Alpha Vantage's News & Sentiment API provides ticker-specific sentiment scores for recent articles. Positive scores indicate more positive news sentiment, negative scores indicate more negative sentiment, and scores near zero are more neutral.

The API key should be stored in Colab Secrets as `ALPHA_VANTAGE_API_KEY` rather than typed directly into the notebook.


In [ ]:
from google.colab import userdata
import requests
import time

alpha_key = userdata.get("ALPHA_VANTAGE_API_KEY")

all_sentiment = []

for ticker in tickers:
    print(f"Pulling sentiment for {ticker}...")

    params = {
        "function": "NEWS_SENTIMENT",
        "tickers": ticker,
        "apikey": alpha_key,
        "limit": 50
    }

    response = requests.get("https://www.alphavantage.co/query", params=params)
    data = response.json()

    if "feed" not in data:
        print(f"No feed returned for {ticker}: {data}")
        continue

    for article in data.get("feed", []):
        for ticker_info in article.get("ticker_sentiment", []):
            if ticker_info["ticker"] == ticker:
                all_sentiment.append({
                    "ticker": ticker,
                    "date": article["time_published"][:8],
                    "title": article["title"],
                    "source": article["source"],
                    "url": article["url"],
                    "ticker_sentiment_score": float(ticker_info["ticker_sentiment_score"]),
                    "ticker_sentiment_label": ticker_info["ticker_sentiment_label"],
                    "relevance_score": float(ticker_info["relevance_score"])
                })

    time.sleep(12)

sentiment_df = pd.DataFrame(all_sentiment)
sentiment_df.head()


In [ ]:
sentiment_df["date"] = pd.to_datetime(sentiment_df["date"], format="%Y%m%d")

daily_sentiment = sentiment_df.groupby(["ticker", "date"]).agg(
    avg_sentiment_score=("ticker_sentiment_score", "mean"),
    avg_relevance_score=("relevance_score", "mean"),
    article_count=("title", "count")
).reset_index()

daily_sentiment


In [ ]:
sentiment_df.to_csv("data/alpha_vantage_news_sentiment_raw.csv", index=False)
daily_sentiment.to_csv("data/daily_sentiment_scores.csv", index=False)

daily_sentiment["ticker"].value_counts()


## 8. Combining Stock and Sentiment Data

Because the free Alpha Vantage news endpoint mostly provides recent articles, this notebook uses the latest available sentiment score for each company as a current market sentiment feature.


In [ ]:
latest_sentiment = daily_sentiment.sort_values("date").groupby("ticker").tail(1)

latest_sentiment = latest_sentiment[[
    "ticker",
    "avg_sentiment_score",
    "avg_relevance_score",
    "article_count"
]]

stock_df = pd.read_csv("data/stock_features.csv")
stock_df["date"] = pd.to_datetime(stock_df["date"])

stock_df = stock_df.merge(latest_sentiment, on="ticker", how="left")

stock_df["avg_sentiment_score"] = stock_df["avg_sentiment_score"].fillna(0)
stock_df["avg_relevance_score"] = stock_df["avg_relevance_score"].fillna(0)
stock_df["article_count"] = stock_df["article_count"].fillna(0)

stock_df.head()


In [ ]:
stock_df.to_csv("data/stock_features_with_sentiment.csv", index=False)


In [ ]:
sentiment_summary = latest_sentiment.sort_values("avg_sentiment_score", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(sentiment_summary["ticker"], sentiment_summary["avg_sentiment_score"])
plt.title("Latest Average News Sentiment by Company")
plt.xlabel("Ticker")
plt.ylabel("Average Sentiment Score")
plt.axhline(0, color="black", linewidth=1)
plt.tight_layout()
plt.savefig("output/latest_sentiment_by_company.png", dpi=300, bbox_inches="tight")
plt.show()


## 9. Prediction Target

The target variable is whether the stock closes higher 7 trading days later. A value of `1` means the stock moved up, and a value of `0` means it did not move up.


In [ ]:
stock_df = pd.read_csv("data/stock_features_with_sentiment.csv")
stock_df["date"] = pd.to_datetime(stock_df["date"])
stock_df = stock_df.sort_values(["ticker", "date"])

stock_df["future_close_7d"] = stock_df.groupby("ticker")["close"].shift(-7)
stock_df["future_return_7d"] = (stock_df["future_close_7d"] / stock_df["close"]) - 1
stock_df["target_up_7d"] = (stock_df["future_return_7d"] > 0).astype(int)

stock_df[[
    "date", "ticker", "close", "future_close_7d",
    "future_return_7d", "target_up_7d"
]].head(10)


In [ ]:
stock_df.to_csv("data/model_ready_stock_data.csv", index=False)


## 10. Next Steps

The next step is to train and evaluate a simple classification model. The model will use stock trend features, volatility, volume change, and sentiment features to predict `target_up_7d`.
